In [18]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/icc_rules.pdf")

documents = loader.load() 

In [19]:
print(len(documents))
print(type(documents))

44
<class 'list'>


In [20]:
documents[43].page_content

'44 \n \n5.2.10 The Match Referee shall have the discretion to announce the substance of his/her decision \nprior to the issue of the written reasoned decision referred to in Article 5.2.9.    \n5.2.11 A copy of the written reasoned decision will be provided to the Player or Player Support \nPersonnel, the CEO of the Player or Player Suppo rt Personnel’s National Cricket \nFederation, and the ICC’s Cricket Operations Manager. \n5.2.12 Subject only to the rights of appeal under Article 8, the Match Referee’s or Judicial \nCommissioner’s decision shall be the full, final and complete disposition of the matter and \nwill be binding on all parties. \n… \n \nARTICLE 8 APPEALS  \n \n- Article 8.2.3 - “48 hours” is replaced with “24 hours”. \n- Article 8.2.3.2 – “48 hours” is replaced with “24 hours”.  \n- Article 8.2.3.3 – “Articles 5.1.2 to 5.1.11” is replaced with “the amended Articles 5.1 and 5.2”.  \n- Article 8.2.3.5 – “seven days” is replaced with “48 hours” . \n- Article 8.3.3 – “seve

In [21]:
documents[0].metadata

{'producer': 'Microsoft® Word for Microsoft 365',
 'creator': 'Microsoft® Word for Microsoft 365',
 'creationdate': '2023-07-26T16:34:54+04:00',
 'title': 'The World Anti-Doping Code',
 'author': 'WADA',
 'moddate': '2023-07-26T16:34:54+04:00',
 'source': 'data/icc_rules.pdf',
 'total_pages': 44,
 'page': 0,
 'page_label': '1'}

@ here we are converting documents  into samll chuncks , now pdf hass 44 pages those convert into chunks , with overlapping to maintain no data should be loss
     

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks=text_splitter.split_documents(documents)

In [23]:
len(chunks)

340

In [24]:
chunks[0].page_content

'1 \n \n \nThe International Cricket Council \n  \nCode of Conduct for Players and Player \nSupport Personnel \n \n  \n \n \n \nEffective as from 16 June 2023 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nFor information regarding the Code of Conduct, please contact: \n \nThe ICC’s Cricket Operations Department  \nInternational Cricket Council \nPO Box 500 070 \nDubai, United Arab Emirates \n \nTel (switchboard):  +971 4 382 8800 \nFacsimile:   +971 4 382 8600'

In [25]:
chunks[0].metadata

{'producer': 'Microsoft® Word for Microsoft 365',
 'creator': 'Microsoft® Word for Microsoft 365',
 'creationdate': '2023-07-26T16:34:54+04:00',
 'title': 'The World Anti-Doping Code',
 'author': 'WADA',
 'moddate': '2023-07-26T16:34:54+04:00',
 'source': 'data/icc_rules.pdf',
 'total_pages': 44,
 'page': 0,
 'page_label': '1'}

we have convert these 341 chunks into something the computer can compare mathematically so, we have to convert these chunks into embeddings , embeddings means vectors(numerical representation). we know that macahine can only understand numbers not words

Now we are going to use hugging-face sentence transformer which is used to conevrt chunks into embeddings

In [26]:
from langchain_huggingface import HuggingFaceEmbeddings

In [27]:
embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

we are testing one chunk  first , then convert into embeddings 

In [28]:
embedding = embedding_model.embed_query(chunks[0].page_content)

print(type(embedding))
print(len(embedding))
print(embedding[:10])

<class 'list'>
384
[-0.017091169953346252, -0.01897810585796833, -0.0949784442782402, -0.0724133625626564, -0.009258410893380642, 0.09715979546308517, -0.016708945855498314, 0.01647992804646492, -0.03611759468913078, 0.10048913210630417]


yeah the chunk is correct now , we can convert all chunks into embeddings 

In [29]:
texts = []

for chunk in chunks:
    texts.append(chunk.page_content)

In [30]:
type(texts)

list

In [31]:
type(texts[0])

str

now all chunks converted into strings because embedding model accepts only strings as input , ican not generate embeddings from documents .

also in above we only taken page-contents beacuse meta-data only useful when user sends query to check that

In [32]:
embeddings=embedding_model.embed_documents(texts)

In [33]:
embedding[:10]

[-0.017091169953346252,
 -0.01897810585796833,
 -0.0949784442782402,
 -0.0724133625626564,
 -0.009258410893380642,
 0.09715979546308517,
 -0.016708945855498314,
 0.01647992804646492,
 -0.03611759468913078,
 0.10048913210630417]

In [34]:
len(embedding)

384

In [35]:
type(embedding)

list

creating vector database and store both embeddings and documents.
because if we give only vectors there is no meaning for those vectors to return while user asks query so that if we also passs the documents then langchain map the vector with respected documents , 
when user asks  query it converts into vectors and search for best in 341 vectors then maps to documents and sent to LLM then output will be generated

In [36]:
from langchain_community.vectorstores import FAISS

In [37]:
vector_store=FAISS.from_documents(  
    documents=chunks,
    embedding=embedding_model
)

In [38]:
vector_store

Now we have sucessfully created vector Database , Data ingestion pipeline is completed now we have to build retervial pipe line 


In [39]:
results = vector_store.similarity_search(
    "What is the penalty for slow over-rate?"
)

In [40]:
results

[Document(id='8f797025-38ba-441a-8db1-400925528946', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-07-26T16:34:54+04:00', 'title': 'The World Anti-Doping Code', 'author': 'WADA', 'moddate': '2023-07-26T16:34:54+04:00', 'source': 'data/icc_rules.pdf', 'total_pages': 44, 'page': 37, 'page_label': '38'}, page_content='Operations Department) within 18 hours of the close of the day’s play in the relevant \nInternational Match or prior to the start of the following day’s play, whichever is the sooner;   \n \n3.2.2 thereafter, the Match Referee shall promptly consult with the Umpires and shall be entitled, \nafter such consultation, to make such amend ments to the actual over rate calculation as \nhe/she deems appropriate in the circumstances to reflect those circumstances that are'),
 Document(id='a27269cd-df8d-4cc7-b113-0707a41ee156', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Micr

In [41]:
len(results)

4

see the above output it given 4 related chunks to query

In [42]:
type(results)

list

In [43]:
type(results[0])

langchain_core.documents.base.Document

In [44]:
vector_store.similarity_search("What punishment is given if the bowling team is too slow?")

[Document(id='165f5b07-d928-49a3-bc2e-f2912913e825', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-07-26T16:34:54+04:00', 'title': 'The World Anti-Doping Code', 'author': 'WADA', 'moddate': '2023-07-26T16:34:54+04:00', 'source': 'data/icc_rules.pdf', 'total_pages': 44, 'page': 31, 'page_label': '32'}, page_content='with the Code of Conduct.   \nARTICLE 10 SANCTIONS AND COSTS ASSESSED AGAINST NATIONAL CRICKET'),
 Document(id='397dcdac-6685-4c70-81ed-824ef2c4f2b3', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-07-26T16:34:54+04:00', 'title': 'The World Anti-Doping Code', 'author': 'WADA', 'moddate': '2023-07-26T16:34:54+04:00', 'source': 'data/icc_rules.pdf', 'total_pages': 44, 'page': 7, 'page_label': '8'}, page_content='thrown; and (iv) the distance from which the ball/object was thrown. \n \nLevel 1 ✓ \nLevel 2 ✓ \nLevel 

Suppose I ask:

"What punishment is given if the bowling team is too slow?"

Notice something?

The words are completely different.

The PDF might say:

"Penalty for slow over-rate"

There is no exact keyword match.


FAISS itself is not the retriever.

FAISS is just the vector database.

The Retriever is a wrapper around FAISS that makes it easy to fetch relevant documents.

Think of it like this:

FAISS
   │
   │  (stores vectors)
   ▼
Retriever
   │
   │  (retrieves relevant documents)
   ▼
LLM

### Convert FAISS into a Retriever

In [45]:
retriever=vector_store.as_retriever()

In [46]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002DA26026B50>, search_kwargs={})

In [47]:
retriever.invoke(
    "What is the penalty for slow over-rate?"
)

[Document(id='8f797025-38ba-441a-8db1-400925528946', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-07-26T16:34:54+04:00', 'title': 'The World Anti-Doping Code', 'author': 'WADA', 'moddate': '2023-07-26T16:34:54+04:00', 'source': 'data/icc_rules.pdf', 'total_pages': 44, 'page': 37, 'page_label': '38'}, page_content='Operations Department) within 18 hours of the close of the day’s play in the relevant \nInternational Match or prior to the start of the following day’s play, whichever is the sooner;   \n \n3.2.2 thereafter, the Match Referee shall promptly consult with the Umpires and shall be entitled, \nafter such consultation, to make such amend ments to the actual over rate calculation as \nhe/she deems appropriate in the circumstances to reflect those circumstances that are'),
 Document(id='a27269cd-df8d-4cc7-b113-0707a41ee156', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Micr

now onwards instead of useing vector_store we can use simple retriver.invoke("Query")

### we aer using retriver because it acts like a searching interface

### Last stage of RAG pipeline

In [48]:
print("Gemini OK")


Gemini OK


In [49]:
from dotenv import load_dotenv

load_dotenv()

True

In [50]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

c:\Users\sathi\Desktop\RAG_model\venv\Lib\site-packages\langchain_google_genai\chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


In [51]:
response = llm.invoke("Say only the word: Connected")

print(response.content)

Connected


In [52]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [53]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [54]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an AI assistant.

Answer the user's question using ONLY the provided context.

Context:
{context}

Question:
{question}
""")

# we got error in output that fill answer was not generatedd so, that we are increasing the chunk size 

In [55]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 10}
)

In [56]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
response = rag_chain.invoke(
    "What "
)

print(response)

This information is not provided in the context.


In [ ]:
response = rag_chain.invoke(
    "can you explain  about injury rues"
)

print(response)

I am sorry, but the provided context does not contain information about "Disobeying an Umpire’s instruction during an International Match."


In [62]:
rag_chain.invoke("Disobeying an Umpire’s instruction during an International Match.")

'The provided context does not explicitly state whether "Disobeying an Umpire’s instruction during an International Match" is an offense. It details other offenses related to Umpires, such as inappropriate physical contact, deliberate attempts to deceive an Umpire, and intimidation of an Umpire or Match Referee.'

### Phase 2 

Phase 1 ✅
PDF Loader
Chunking
Embeddings
FAISS
Retriever
Gemini
Basic RAG

──────────────

Phase 2 🚀
Save FAISS Index
Load FAISS Index
Chat Interface
Memory
Advanced Retrieval
Deployment

In [63]:
vector_store.save_local("faiss_index")

after runnig above line now our vector database is in our disc. so when even i cose my laptop again , at that no need to run all the cells ike chunking , embedding , databases , documents , now we can retrive our answer through vector data base which is present in the local 

In [64]:
import os

os.listdir()

['.env',
 'app.ipynb',
 'data',
 'faiss_index',
 'requirements.txt',
 'test.py',
 'venv']

In [66]:
from langchain_community.vectorstores import FAISS

In [68]:
loaded_vector_store=FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
    )

In [94]:
retriever = loaded_vector_store.as_retriever(
    search_kwargs={"k": 10}
)



In [95]:
docs=retriever.invoke(
    "what is the penality for slow over-rate ?"
)

In [96]:
len(docs)

10

In [97]:
docs

[Document(id='a27269cd-df8d-4cc7-b113-0707a41ee156', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-07-26T16:34:54+04:00', 'title': 'The World Anti-Doping Code', 'author': 'WADA', 'moddate': '2023-07-26T16:34:54+04:00', 'source': 'data/icc_rules.pdf', 'total_pages': 44, 'page': 34, 'page_label': '35'}, page_content='he/she may be assisted in the administrative performance of his/her duties under this Code of Conduct by \nany official ‘Match Manager’ who may be appointed to officiate at such International Match.        \n \nMinimum Over Rate.  As defined in Appendix 2 of this Code of Conduct. \n \nMinimum Over Rate Offence. Any of the offences described in Articles 2.22.1 – 2.22.2. \n \nMinor Over Rate Offence.  As defined in Article 2.22.1.'),
 Document(id='8f797025-38ba-441a-8db1-400925528946', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'cre